In [2]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [3]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [4]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "MOBILE": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3003, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13148, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28015, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38425, 38)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154252, 41)
Loaded: raw_hq_icmas_products.csv -> (114982, 94)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50307, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733701, 38)
Loaded: raw_hq_armas_receivable.csv -> (2232, 20)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276183, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13875, 32)


In [6]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/online/lazada"

In [47]:
import pandas as pd
import glob
import os

files = glob.glob(os.path.join(folder, "*.xlsx"))

df_lazada_raw = pd.concat(
    [
        pd.read_excel(
            f,
            sheet_name="Transaction Overview",
            header=0,        # explicitly say header is first row
            dtype={
                "Seller SKU": "string",
                "Transaction Number": "string",
                "Reference": "string",
            }
        )
        for f in files
    ],
    ignore_index=True
)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

In [48]:
df_lazada_raw.columns

Index(['Transaction Date', 'Transaction Type', 'Fee Name',
       'Transaction Number', 'Details', 'Seller SKU', 'Lazada SKU', 'Amount',
       'VAT in Amount', 'WHT Amount', 'WHT included in Amount', 'Statement',
       'Paid Status', 'Order No.', 'Order Item No.', 'Order Item Status',
       'Shipping Provider', 'Shipping Speed', 'Shipment Type', 'Reference',
       'Comment', 'PaymentRefId', 'ShortCode'],
      dtype='object')

In [49]:
df = df_lazada_raw.copy()

# ensure numeric
df["Amount"] = (
    pd.to_numeric(df["Amount"], errors="coerce")
)

# composite key
df["sku_txn_key"] = (
    df["Seller SKU"].astype("string") + "|" +
    df["Transaction Number"].astype("string")
)

# aggregation
df_grouped = (
    df
    .groupby(["Seller SKU", "Transaction Number"], dropna=False)
    .agg(
        amount_signed = ("Amount", "sum"),
        amount_positive = ("Amount", lambda s: s[s > 0].sum()),
        amount_negative = ("Amount", lambda s: s[s < 0].sum()),
        line_count = ("Amount", "size")
    )
    .reset_index()
)

In [50]:
df_grouped

,Seller SKU,Transaction Number,amount_signed,amount_positive,amount_negative,line_count
0,05052418,1080221530476898,649.85,798.0,-148.15,6
1,05052418,1080221530576898,649.85,798.0,-148.15,6
2,07010524,1085363326136892,123.08,145.0,-21.92,4
3,08010644,1085698553112609,3208.74,3790.0,-581.26,5
4,08017381,1081633748889005,532.67,650.0,-117.33,5
...,...,...,...,...,...,...
409,5088996259-1709591764409-0,1071723465828335,1453.16,1640.0,-186.84,3
410,5968962800-1764129397314-0,1084478547990365,1212.63,1480.0,-267.37,5
411,<NA>,TH1JI09A4U-2026-0203##wrongWeightAdjustment,-99.00,0.0,-99.00,1
412,<NA>,TH1K374LGJ-2026-0204##wrongWeightAdjustment,-13.00,0.0,-13.00,1


In [38]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()

In [39]:
df_out = df_grouped.merge(
    df_simas[["PO", "BILLNO"]],
    how="left",
    left_on="Reference",
    right_on="PO"
)

In [40]:
txn = "108090295836812"

df_found = df_simas[
    df_simas["PO"].astype("string") == txn
]

In [41]:
df_found

,ID,JOURMODE,JOURTYPE,JOURDATE,JOURNO,JOURTIME,DEPTNO,BOOKNO,BILLTYPE,BILLDATE,...,NOTENO,VOUCDATE1,VOUCNO1,VOUCDATE2,VOUCNO2,POSTED1,POSTED2,REMARKS,CANCELED,DONE
270034,533631,1,SJ,2026-02-03,NaN,08:34,1,2,1,2026-02-03,...,NaN,NaN,NaN,2026-02-11,RVI6902-017,N,N,##0000000000001,N,N


In [42]:
df_out

,Seller SKU,Reference,amount_signed,amount_positive,amount_negative,line_count,PO,BILLNO
0,05052418,1080221530476898,649.85,798.0,-148.15,6,NaN,<NA>
1,05052418,1080221530576898,649.85,798.0,-148.15,6,NaN,<NA>
2,07010524,1085363326136892,123.08,145.0,-21.92,4,NaN,<NA>
3,08010644,1085698553112609,3208.74,3790.0,-581.26,5,NaN,<NA>
4,08017381,1081633748889005,532.67,650.0,-117.33,5,NaN,<NA>
...,...,...,...,...,...,...,...,...
406,35050126,1081976996526077,571.17,675.0,-103.83,5,NaN,<NA>
407,35050150,1084565350216011,273.64,325.0,-51.36,4,NaN,<NA>
408,5088996259-1709591764409-0,1071723465828335,1453.16,1640.0,-186.84,3,NaN,<NA>
409,5968962800-1764129397314-0,1084478547990365,1212.63,1480.0,-267.37,5,NaN,<NA>


In [ ]:
df_out["deduction_pct"] = (
    (df_out["amount_full"] - df_out["after_fee"]) / df_out["amount_full"] * 100
).round(2)

In [ ]:
df_out

,date,shopee_no,amount_full,after_fee,PO,BILLNO,deduction_pct
0,2026-01-29,260129BBGQVS7P,540,462,260129BBGQVS7P,TAD6901-621,14.44
1,2026-01-30,260130CPHQEP8N,1980,1702,260130CPHQEP8N,TAD6901-627,14.04
2,2026-01-25,2601250Y0M8BJ9,3450,2959,2601250Y0M8BJ9,TAD6901-526,14.23
3,2026-01-25,2601250AATYBEF,3500,3000,2601250AATYBEF,TAD6901-524,14.29
4,2026-01-27,2601275AY4Q8CS,3990,3539,2601275AY4Q8CS,TAD6901-569,11.30
...,...,...,...,...,...,...,...
94,2026-02-21,260221B0TW90P3,590,523,260221B0TW90P3,TAD6902-484,11.36
95,2026-02-21,260221A59TT49G,2900,2572,260221A59TT49G,TAD6902-482,11.31
96,2026-02-22,260222CWEM5H8G,3450,2969,260222CWEM5H8G,TAD6902-485,13.94
97,2026-02-26,260226PXGABAGU,6900,5599,260226PXGABAGU,TAD6902-582,18.86


In [ ]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "03_curated",
    f"shopee.csv"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [ ]:
df_out[["BILLNO","deduction_pct"]].to_csv(folder, index=False, encoding="utf-8-sig")